In [2]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import matplotlib

with open('/content/shakespeare.txt') as input_file:
    text_data = input_file.read()

chars = sorted(list(set(text_data)))

torch.set_default_device('cuda')

def encode(str):
    out = []
    for i in str: out.append(chars.index(i))
    return out
def decode(list):
    out = ''
    for i in list: out = f'{out}{chars[i]}'
    return out

block_size = 8
batch_size = 4
embedding_dims = 32
head_size = 16

torch.manual_seed(1234)

split_rate = 0.9
split_idx = int(split_rate * len(text_data))
train_data = text_data[:split_idx]
val_data = text_data[split_idx:]

val_step = 0

def get_batch(batchsize, blocksize, data):
    idx = torch.randint(len(data) - blocksize, (batchsize, ))
    x = torch.stack([torch.tensor(encode(data[i:i+blocksize])) for i in idx])
    y = torch.stack([torch.tensor(encode(data[i+1:i+blocksize+1])) for i in idx])
    return x,y

class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(embedding_dims, head_size, bias=False)
        self.value = nn.Linear(embedding_dims, head_size, bias=False)
        self.query = nn.Linear(embedding_dims, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
    def forward(self, ins):
        k = self.key(ins)
        q = self.query(ins)
        wei = q @ k.transpose(-2, -1)
        wei_norm = wei * head_size **-0.5
        wei_norm = wei_norm.masked_fill(self.tril[:ins.shape[1], :ins.shape[1]] == 0, float('-inf')) 
        wei_norm = F.softmax(wei_norm, dim=-1)
        v = self.value(ins)
        out = wei_norm @ v

        return out


class MultiHeadAttention(nn.Module):
    def __init__(self, n_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(n_heads)])
    def forward(self, x):
        return torch.cat([h(x) for h in self.heads], dim=-1)

class FeedForward(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(emb_dim, emb_dim),
            nn.ReLU()
        )
    def forward(self, x):
        return self.net(x)

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, embedding_dims)
        self.positional_encoding = nn.Embedding(block_size, embedding_dims)
        self.sa_heads = MultiHeadAttention(4, embedding_dims//4)
        self.lm_head = nn.Linear(embedding_dims, len(chars))
        self.ff = FeedForward(embedding_dims)
    def forward(self, idx, targets=None):
        token_embeddings = self.token_embedding_table(idx)
        pos_embeddings = self.positional_encoding(torch.arange(idx.shape[1]))
        x = token_embeddings + pos_embeddings
        x = self.sa_heads(x)
        x = self.ff(x)
        
        logits = self.lm_head(x)
        if targets is not None: 
            B,T,C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
            return logits, loss
        else:
            return logits, None
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=1)
            next_idx = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, next_idx), dim=1)
        return idx



model = BigramLanguageModel(len(chars))
batch = get_batch(batch_size, block_size ,train_data)

optim = torch.optim.AdamW(model.parameters(), lr=1e-3)

total_steps = 10000
for step in range(total_steps):
    x, y = get_batch(4,8,train_data)
    logits, loss = model(x,y)
    optim.zero_grad(set_to_none=True)
    loss.backward()
    if not step%1000: print(f"{step}/{total_steps}: {loss.item()}")
    optim.step()


xval = torch.tensor([encode(val_data[:-1])])
yval = torch.tensor([encode(val_data[1:])])
loss_val = 0
for _ in range(5000):
    batch = get_batch(batch_size, block_size, val_data)
    _, loss = model(batch[0], batch[1])
    loss_val += loss.item()
    val_step += 1
print(f'val loss: {loss_val/5000}')


ctx = torch.zeros(1,1, dtype=torch.long)
print(decode(model.generate(ctx, 1000)[0].tolist()))

0/10000: 4.141216278076172
1000/10000: 2.5921568870544434
2000/10000: 2.4818978309631348
3000/10000: 2.787156343460083
4000/10000: 2.3419909477233887
5000/10000: 2.497432231903076
6000/10000: 2.4891693592071533
7000/10000: 2.201991081237793
8000/10000: 3.0325355529785156
9000/10000: 2.5085363388061523
val loss: 2.298838168859482

Thorge,
Willed sueard; if miss fre-durd, woow the.

IRI:
Miats, wiy, shis Mut thand fu pur, hit tish, me,
Tot, ving love troces in a buve yoece fowgthy Atstry uvius noureot my outcopen tre your laky fuare at; asteto wit, bet:
Tor
Aigheesph'd drecrow to sou ish chis Sof.
DENNUS:
Butme, ipndutluges, qnI asstis tameand-GNoutih dree.


BED:
Twowngo eproft;
BuiqIs
Sid acaltough in to hid ouuimt,
Dom, rats.

Winetit, I eperwiand beeerise spane.

BET:
Oy prieg, preve wee
As:
Af I ar froufefratb ought travuk butsu, comecou wans peraten! E INID:
Fyord'
Dover by nothe not sed I be heis it lesrriceg
Ascl barind teoiglar.
I
Toser throamfoms band nor.

IUSERCIOt brend cutr

In [4]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import matplotlib

with open('/content/shakespeare.txt') as input_file:
    text_data = input_file.read()

chars = sorted(list(set(text_data)))

torch.set_default_device('cuda')

def encode(str):
    out = []
    for i in str: out.append(chars.index(i))
    return out
def decode(list):
    out = ''
    for i in list: out = f'{out}{chars[i]}'
    return out

block_size = 8
batch_size = 4
embedding_dims = 32
head_size = 16

torch.manual_seed(1234)

split_rate = 0.9
split_idx = int(split_rate * len(text_data))
train_data = text_data[:split_idx]
val_data = text_data[split_idx:]

val_step = 0

def get_batch(batchsize, blocksize, data):
    idx = torch.randint(len(data) - blocksize, (batchsize, ))
    x = torch.stack([torch.tensor(encode(data[i:i+blocksize])) for i in idx])
    y = torch.stack([torch.tensor(encode(data[i+1:i+blocksize+1])) for i in idx])
    return x,y

class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(embedding_dims, head_size, bias=False)
        self.value = nn.Linear(embedding_dims, head_size, bias=False)
        self.query = nn.Linear(embedding_dims, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
    def forward(self, ins):
        k = self.key(ins)
        q = self.query(ins)
        wei = q @ k.transpose(-2, -1)
        wei_norm = wei * head_size **-0.5
        wei_norm = wei_norm.masked_fill(self.tril[:ins.shape[1], :ins.shape[1]] == 0, float('-inf')) 
        wei_norm = F.softmax(wei_norm, dim=-1)
        v = self.value(ins)
        out = wei_norm @ v

        return out


class MultiHeadAttention(nn.Module):
    def __init__(self, n_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(n_heads)])
        self.proj = nn.Linear(embedding_dims, embedding_dims)
    def forward(self, x):
        return self.proj(torch.cat([h(x) for h in self.heads], dim=-1))

class FeedForward(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(emb_dim, 4*emb_dim),
            nn.ReLU(),
            nn.Linear(4*emb_dim, emb_dim) # projection
        )
    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd, n_heads):
        super().__init__()
        head_size = n_embd // n_heads
        self.sa = MultiHeadAttention(n_heads, head_size)
        self.ff = FeedForward(n_embd)
    def forward(self, x):
        x = x + self.sa(x)
        x = x + self.ff(x)
        return x

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, embedding_dims)
        self.positional_encoding = nn.Embedding(block_size, embedding_dims)
        self.blocks = nn.Sequential(
            Block(embedding_dims, n_heads=4)
        )
        self.lm_head = nn.Linear(embedding_dims, len(chars))
    def forward(self, idx, targets=None):
        token_embeddings = self.token_embedding_table(idx)
        pos_embeddings = self.positional_encoding(torch.arange(idx.shape[1]))
        x = token_embeddings + pos_embeddings
        x = self.blocks(x)
        
        logits = self.lm_head(x)
        if targets is not None: 
            B,T,C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
            return logits, loss
        else:
            return logits, None
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=1)
            next_idx = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, next_idx), dim=1)
        return idx



model = BigramLanguageModel(len(chars))
batch = get_batch(batch_size, block_size ,train_data)

optim = torch.optim.AdamW(model.parameters(), lr=1e-3)

total_steps = 10000
for step in range(total_steps):
    x, y = get_batch(4,8,train_data)
    logits, loss = model(x,y)
    optim.zero_grad(set_to_none=True)
    loss.backward()
    if not step%1000: print(f"{step}/{total_steps}: {loss.item()}")
    optim.step()


xval = torch.tensor([encode(val_data[:-1])])
yval = torch.tensor([encode(val_data[1:])])
loss_val = 0
for _ in range(5000):
    batch = get_batch(batch_size, block_size, val_data)
    _, loss = model(batch[0], batch[1])
    loss_val += loss.item()
    val_step += 1
print(f'val loss: {loss_val/5000}')


ctx = torch.zeros(1,1, dtype=torch.long)
print(decode(model.generate(ctx, 1000)[0].tolist()))

0/10000: 4.502405166625977
1000/10000: 2.559091091156006
2000/10000: 2.4475698471069336
3000/10000: 2.422027587890625
4000/10000: 2.2621073722839355
5000/10000: 2.130000114440918
6000/10000: 2.5989937782287598
7000/10000: 2.1633009910583496
8000/10000: 2.6447832584381104
9000/10000: 2.4433488845825195
val loss: 2.2438808656454086

And ut couds, aris if mizs fomery youst,
Ward.

IRe:
Muave, wouccurses us thate fords ith fore hat whiTh; whilr love to his in a fue of evedfurgtingm sure uvery noureot my it copenter fod, all your fane staugeto wity sutiett God heespe'r dreceove of rote hen is Soff wid.
Sonfe my tird ut unve, qul asstiow a thous loutiocerme.
Sriedl ow word epfort;
Buinks hidl madiough ind of now suith befellrets.

Wity it, I eperweat; beeerised, tue.
Sove it it ferd prive weereste;
Buvar froife ratb oughte praknd I sut comecords nove walen! E If swellcow'll Yordst nottA not sed I be my shis learriees
Ancl bacindlteoigllr.
I
Postr the hefoms OET rown. But, at sto
bron's hrome

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import matplotlib

with open('/content/shakespeare.txt') as input_file:
    text_data = input_file.read()

chars = sorted(list(set(text_data)))

torch.set_default_device('cuda')

def encode(str):
    out = []
    for i in str: out.append(chars.index(i))
    return out
def decode(list):
    out = ''
    for i in list: out = f'{out}{chars[i]}'
    return out

block_size = 8
batch_size = 4
embedding_dims = 32
head_size = 16

torch.manual_seed(1234)

split_rate = 0.9
split_idx = int(split_rate * len(text_data))
train_data = text_data[:split_idx]
val_data = text_data[split_idx:]

val_step = 0

def get_batch(batchsize, blocksize, data):
    idx = torch.randint(len(data) - blocksize, (batchsize, ))
    x = torch.stack([torch.tensor(encode(data[i:i+blocksize])) for i in idx])
    y = torch.stack([torch.tensor(encode(data[i+1:i+blocksize+1])) for i in idx])
    return x,y

class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(embedding_dims, head_size, bias=False)
        self.value = nn.Linear(embedding_dims, head_size, bias=False)
        self.query = nn.Linear(embedding_dims, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
    def forward(self, ins):
        k = self.key(ins)
        q = self.query(ins)
        wei = q @ k.transpose(-2, -1)
        wei_norm = wei * head_size **-0.5
        wei_norm = wei_norm.masked_fill(self.tril[:ins.shape[1], :ins.shape[1]] == 0, float('-inf')) 
        wei_norm = F.softmax(wei_norm, dim=-1)
        v = self.value(ins)
        out = wei_norm @ v

        return out


class MultiHeadAttention(nn.Module):
    def __init__(self, n_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(n_heads)])
        self.proj = nn.Linear(embedding_dims, embedding_dims)
    def forward(self, x):
        return self.proj(torch.cat([h(x) for h in self.heads], dim=-1))

class FeedForward(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(emb_dim, 4*emb_dim),
            nn.ReLU(),
            nn.Linear(4*emb_dim, emb_dim) # projection
        )
    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd, n_heads):
        super().__init__()
        head_size = n_embd // n_heads
        self.sa = MultiHeadAttention(n_heads, head_size)
        self.ff = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)
    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, embedding_dims)
        self.positional_encoding = nn.Embedding(block_size, embedding_dims)
        self.blocks = nn.Sequential(
            Block(embedding_dims, n_heads=4)
        )
        self.lm_head = nn.Linear(embedding_dims, len(chars))
    def forward(self, idx, targets=None):
        token_embeddings = self.token_embedding_table(idx)
        pos_embeddings = self.positional_encoding(torch.arange(idx.shape[1]))
        x = token_embeddings + pos_embeddings
        x = self.blocks(x)
        
        logits = self.lm_head(x)
        if targets is not None: 
            B,T,C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
            return logits, loss
        else:
            return logits, None
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=1)
            next_idx = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, next_idx), dim=1)
        return idx



model = BigramLanguageModel(len(chars))
batch = get_batch(batch_size, block_size ,train_data)

optim = torch.optim.AdamW(model.parameters(), lr=1e-3)

total_steps = 10000
for step in range(total_steps):
    x, y = get_batch(4,8,train_data)
    logits, loss = model(x,y)
    optim.zero_grad(set_to_none=True)
    loss.backward()
    if not step%1000: print(f"{step}/{total_steps}: {loss.item()}")
    optim.step()


xval = torch.tensor([encode(val_data[:-1])])
yval = torch.tensor([encode(val_data[1:])])
loss_val = 0
for _ in range(5000):
    batch = get_batch(batch_size, block_size, val_data)
    _, loss = model(batch[0], batch[1])
    loss_val += loss.item()
    val_step += 1
print(f'val loss: {loss_val/5000}')


ctx = torch.zeros(1,1, dtype=torch.long)
print(decode(model.generate(ctx, 1000)[0].tolist()))